# Data Cleaning Notebook
## Fraud Detection Project - Task 1a

This notebook handles data cleaning operations including:
- Loading raw datasets
- Handling missing values
- Removing duplicates
- Converting data types
- Saving cleaned data

In [18]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')

from data_loader import DataLoader
from data_cleaner import DataCleaner

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

%matplotlib inline

## 1. Load Raw Data

In [19]:
# Initialize data loader
loader = DataLoader()

# Load datasets
fraud_df = loader.load_fraud_data('../data/raw/Fraud_Data.csv')
ip_mapping_df = loader.load_ip_mapping('../data/raw/IpAddress_to_Country.csv')
creditcard_df = loader.load_creditcard_data('../data/raw/creditcard.csv')

print("Data loaded successfully!")
loader.get_data_summary()

INFO:data_loader:Loading fraud data from ../data/raw/Fraud_Data.csv
INFO:data_loader:Loaded 151112 fraud records
INFO:data_loader:Loading IP mapping data from ../data/raw/IpAddress_to_Country.csv
INFO:data_loader:Loaded 138846 IP mapping records
INFO:data_loader:Loading credit card data from ../data/raw/creditcard.csv
INFO:data_loader:Loaded 284807 credit card transaction records


Data loaded successfully!


{'fraud_data': {'rows': 151112,
  'columns': 11,
  'memory_usage_mb': np.float64(56.89167594909668)},
 'ip_mapping': {'rows': 138846,
  'columns': 3,
  'memory_usage_mb': np.float64(9.978367805480957)},
 'creditcard': {'rows': 284807,
  'columns': 31,
  'memory_usage_mb': np.float64(67.36017990112305)}}

## 2. Inspect Fraud Data

In [20]:
print("Fraud Data Shape:", fraud_df.shape)
fraud_df.head()

Fraud Data Shape: (151112, 11)


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


In [21]:
# Data types
fraud_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   user_id         151112 non-null  int64  
 1   signup_time     151112 non-null  object 
 2   purchase_time   151112 non-null  object 
 3   purchase_value  151112 non-null  int64  
 4   device_id       151112 non-null  object 
 5   source          151112 non-null  object 
 6   browser         151112 non-null  object 
 7   sex             151112 non-null  object 
 8   age             151112 non-null  int64  
 9   ip_address      151112 non-null  float64
 10  class           151112 non-null  int64  
dtypes: float64(1), int64(4), object(6)
memory usage: 12.7+ MB


In [22]:
# Summary statistics
fraud_df.describe()

,user_id,purchase_value,age,ip_address,class
count,151112.000000,151112.000000,151112.000000,1.511120e+05,151112.000000
mean,200171.040970,36.935372,33.140704,2.152145e+09,0.093646
std,115369.285024,18.322762,8.617733,1.248497e+09,0.291336
min,2.000000,9.000000,18.000000,5.209350e+04,0.000000
25%,100642.500000,22.000000,27.000000,1.085934e+09,0.000000
50%,199958.000000,35.000000,33.000000,2.154770e+09,0.000000
75%,300054.000000,49.000000,39.000000,3.243258e+09,0.000000
max,400000.000000,154.000000,76.000000,4.294850e+09,1.000000


## 3. Missing Values Analysis

In [23]:
# Check for missing values
missing_values = fraud_df.isnull().sum()
missing_pct = (missing_values / len(fraud_df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Percentage': missing_pct
})

missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

,Missing Count,Percentage


In [24]:
# Visualize missing values
if missing_values.sum() > 0:
    plt.figure(figsize=(12, 6))
    sns.heatmap(fraud_df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
    plt.title('Missing Values Heatmap - Fraud Data')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in fraud data!")

No missing values found in fraud data!


## 4. Duplicate Detection

In [25]:
# Check for exact duplicates
duplicate_rows = fraud_df.duplicated().sum()
print(f"Exact duplicate rows: {duplicate_rows}")

# Check for duplicate user_ids (could be legitimate)
duplicate_users = fraud_df['user_id'].duplicated().sum()
print(f"Duplicate user_ids: {duplicate_users}")

# Check for duplicate transactions (same user, same purchase time)
duplicate_transactions = fraud_df.duplicated(subset=['user_id', 'purchase_time']).sum()
print(f"Duplicate transactions (same user + time): {duplicate_transactions}")

Exact duplicate rows: 0
Duplicate user_ids: 0
Duplicate transactions (same user + time): 0


## 5. Clean Fraud Data

In [26]:
# Initialize cleaner
cleaner = DataCleaner()

# Handle missing values
fraud_df_clean = cleaner.handle_missing_values(fraud_df, strategy='auto')

# Remove duplicates
fraud_df_clean = cleaner.remove_duplicates(fraud_df_clean)

# Convert date columns
fraud_df_clean = cleaner.convert_data_types(
    fraud_df_clean, 
    datetime_cols=['signup_time', 'purchase_time']
)

# Validate age range (18-100)
fraud_df_clean = cleaner.validate_ranges(
    fraud_df_clean,
    {'age': (18, 100), 'purchase_value': (0, 10000)}
)

print("\n" + cleaner.get_cleaning_report())

INFO:data_cleaner:No missing values found
INFO:data_cleaner:Removed 0 exact duplicate records
INFO:data_cleaner:Converted signup_time to datetime
INFO:data_cleaner:Converted purchase_time to datetime



Data Cleaning Report
1. Removed 0 duplicates
2. Converted signup_time to datetime
3. Converted purchase_time to datetime



In [27]:
print(f"Original shape: {fraud_df.shape}")
print(f"Cleaned shape: {fraud_df_clean.shape}")
print(f"Rows removed: {fraud_df.shape[0] - fraud_df_clean.shape[0]}")

Original shape: (151112, 11)
Cleaned shape: (151112, 11)
Rows removed: 0


## 6. Clean Credit Card Data

In [28]:
print("Credit Card Data Shape:", creditcard_df.shape)
creditcard_df.head()

Credit Card Data Shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [29]:
# Check for missing values in credit card data
cc_missing = creditcard_df.isnull().sum()
print("Missing values in credit card data:")
print(cc_missing[cc_missing > 0] if cc_missing.sum() > 0 else "No missing values")

Missing values in credit card data:
No missing values


In [30]:
# Clean credit card data
cleaner_cc = DataCleaner()

creditcard_df_clean = cleaner_cc.handle_missing_values(creditcard_df, strategy='auto')
creditcard_df_clean = cleaner_cc.remove_duplicates(creditcard_df_clean)

print(f"\nOriginal shape: {creditcard_df.shape}")
print(f"Cleaned shape: {creditcard_df_clean.shape}")

INFO:data_cleaner:No missing values found


INFO:data_cleaner:Removed 1081 exact duplicate records



Original shape: (284807, 31)
Cleaned shape: (283726, 31)


## 7. Inspect IP Mapping Data

In [31]:
print("IP Mapping Data Shape:", ip_mapping_df.shape)
ip_mapping_df.head()

IP Mapping Data Shape: (138846, 3)


,lower_bound_ip_address,upper_bound_ip_address,country
0,16777216.0,16777471,Australia
1,16777472.0,16777727,China
2,16777728.0,16778239,China
3,16778240.0,16779263,Australia
4,16779264.0,16781311,China


In [32]:
# Check for missing values in IP mapping
ip_missing = ip_mapping_df.isnull().sum()
print("Missing values in IP mapping data:")
print(ip_missing[ip_missing > 0] if ip_missing.sum() > 0 else "No missing values")

Missing values in IP mapping data:
No missing values


## 8. Save Cleaned Data

In [33]:
# Save cleaned datasets
fraud_df_clean.to_csv('../data/processed/cleaned_fraud_data.csv', index=False)
creditcard_df_clean.to_csv('../data/processed/cleaned_creditcard.csv', index=False)

print("Cleaned data saved successfully!")
print(f"- fraud_df_clean: {fraud_df_clean.shape}")
print(f"- creditcard_df_clean: {creditcard_df_clean.shape}")

Cleaned data saved successfully!
- fraud_df_clean: (151112, 11)
- creditcard_df_clean: (283726, 31)


## 9. Data Quality Summary

In [34]:
# Create quality report
quality_report = {
    'Dataset': ['Fraud Data', 'Credit Card Data'],
    'Original Rows': [fraud_df.shape[0], creditcard_df.shape[0]],
    'Cleaned Rows': [fraud_df_clean.shape[0], creditcard_df_clean.shape[0]],
    'Rows Removed': [
        fraud_df.shape[0] - fraud_df_clean.shape[0],
        creditcard_df.shape[0] - creditcard_df_clean.shape[0]
    ],
    'Removal %': [
        ((fraud_df.shape[0] - fraud_df_clean.shape[0]) / fraud_df.shape[0]) * 100,
        ((creditcard_df.shape[0] - creditcard_df_clean.shape[0]) / creditcard_df.shape[0]) * 100
    ]
}

quality_df = pd.DataFrame(quality_report)
quality_df

,Dataset,Original Rows,Cleaned Rows,Rows Removed,Removal %
0,Fraud Data,151112,151112,0,0.000000
1,Credit Card Data,284807,283726,1081,0.379555


## Summary

### Data Cleaning Completed

**Fraud Data:**
- Handled missing values
- Removed duplicates
- Converted date columns to datetime format
- Validated age and purchase value ranges

**Credit Card Data:**
- Checked for and handled missing values
- Removed duplicates

**Next Steps:**
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Class Imbalance Handling